# Voice-worker в Google Colab

**Как открыть:** загрузите этот файл `colab.ipynb` в [Google Colab](https://colab.research.google.com/) через **Файл → Загрузить блокнот**, либо положите репозиторий на Drive и откройте ноутбук оттуда.

Подробности см. **`COLAB.md`** в том же репозитории, что и этот ноутбук.

На Colab нужен **только** каталог **voice-worker** (Python). Go-gateway и веб-интерфейс запускаются локально на ПК.

**Код и база хранятся на Google Drive** — ничего не теряется между сессиями.

Структура на Drive (пример):
```
MyDrive/
└── mafia-voice/
    ├── voice-worker/          ← скопируйте сюда только services/voice-server/voice-worker
    └── voice_registry.sqlite  ← опционально (или задаётся VOICE_SERVER_DB)
```

> Скопируйте папку **voice-worker** на Drive **один раз**.

### Порядок при каждом **новом** подключении Colab

1. **Шаг 1** — монтирование Drive.
2. **Шаг 2** — `pip uninstall torchvision` и `pip install -r requirements.txt` (снимает конфликт с образом Colab).
3. **Обязательно:** **Runtime → Restart session** (рестарт среды). Без этого часто остаются ошибки импорта **torch** / **transformers**.
4. Снова **шаг 1** (mount) — после рестарта диск нужно смонтировать заново.
5. **Шаг 3** — переменные окружения (ячейка с `VOICE_SERVER_DB`, `HF_TOKEN`, …). Повторять `pip` после рестарта **не нужно** (пакеты сохраняются в среде).
6. **Шаг 4** — запуск uvicorn + ngrok (последняя ячейка с кодом).

Если вы **не** меняли зависимости и уже делали рестарт в этой сессии, можно начинать с шага 1, затем сразу шаг 3 (переменные) и шаг 4.


In [ ]:
# Шаг 1: монтируем Google Drive (всегда первым)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Шаг 2: снять конфликт torchvision с torch/whisperx и установить зависимости
import os

VOICE_WORKER_ROOT = '/content/drive/MyDrive/mafia-voice/voice-worker'

os.chdir(VOICE_WORKER_ROOT)

!pip uninstall -y torchvision
!pip install -q -r requirements.txt

### ⚠ После установки зависимостей — обязательный рестарт

1. Меню **Runtime** (Среда выполнения) → **Restart session** — перезапустить среду выполнения.
2. После рестарта выполните снова **ячейку «Шаг 1»** (монтирование Google Drive).
3. Дальше — ячейку с **переменными окружения** (следующая после этой) и затем запуск сервера.

**Не повторяйте** ячейку с `pip install`, если только что не начинали сессию с нуля: установленные пакеты сохраняются до конца сессии Colab.

Без рестарта после `pip` часто возникают ошибки вроде несовместимости **torch** и **torchvision** с **whisperx**.

In [ ]:
# Шаг 3: переменные окружения (после рестарта — снова выполните ячейку «Шаг 1» mount)
import os

# Только voice-worker на Drive (не весь voice-server)
VOICE_WORKER_ROOT = '/content/drive/MyDrive/mafia-voice/voice-worker'

assert os.path.isdir(VOICE_WORKER_ROOT), (
    f'Каталог не найден: {VOICE_WORKER_ROOT}\n'
    'Скопируйте services/voice-server/voice-worker в MyDrive/mafia-voice/voice-worker'
)

DB_DIR = '/content/drive/MyDrive/mafia-voice'
os.makedirs(DB_DIR, exist_ok=True)
os.environ['VOICE_SERVER_DB'] = f'{DB_DIR}/voice_registry.sqlite'

os.environ['VOICE_SERVER_DEVICE'] = 'cuda'
os.environ['VOICE_SERVER_API_KEY'] = 'barchik'

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

assert os.environ.get('HF_TOKEN'), 'Задайте HF_TOKEN в Colab Secrets или os.environ'

print('VOICE_WORKER_ROOT:', VOICE_WORKER_ROOT)
print('VOICE_SERVER_DB  :', os.environ['VOICE_SERVER_DB'])

### Шаг 4: где именно запускается Whisper / voice-worker

**Сервис поднимается в следующей ячейке с кодом** (не в отдельном терминале).

Цепочка такая:

1. **`from app.main import app`** — загружается приложение FastAPI из файла **`voice-worker/app/main.py`** (пакет `app` на диске в `VOICE_WORKER_ROOT`).
2. **`uvicorn.Server` + `await server.serve()`** — запускается **HTTP-сервер на порту 8000**. Это и есть «виспер-сервис»: он принимает запросы (`/process_chunk`, `/reset`, …).
3. **Whisper / pyannote** вызываются **внутри обработчиков** при обращении к API (модели часто подгружаются при **первом** запросе с аудио, см. `app/pipeline.py`), а не в момент старта uvicorn.
4. **ngrok** пробрасывает порт 8000 в интернет — по напечатанному `https://…` вы потом указываете **`-voice-url`** в Go на своём ПК.

Ячейка будет выполняться долго («висеть») — пока работает сервер. Остановка: **Stop** или Runtime → Interrupt.


In [ ]:
# Шаг 4: ngrok + uvicorn (публичный URL для gateway на ПК)
# === uvicorn поднимает FastAPI app из app/main.py (Whisper при /process_chunk) ===
import nest_asyncio
nest_asyncio.apply()

import os
import sys

sys.path.insert(0, VOICE_WORKER_ROOT)
os.chdir(VOICE_WORKER_ROOT)

from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url if hasattr(tunnel, 'public_url') else str(tunnel)
print(f">>> Публичный URL (BaseURL для Go): {public_url}")

import uvicorn
from app.main import app

config = uvicorn.Config(app, host='0.0.0.0', port=8000)
server = uvicorn.Server(config)
await server.serve()


Чтобы остановить сервер: **прервите выполнение** ячейки выше (кнопка Stop) или Runtime → Interrupt.

Секрет **NGROK_TOKEN** добавьте в Colab (иконка ключа), как для старого ноутбука Whisper.
